# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MusaGaya/KGaya/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

Method choice: Random Forest Classifier

Random Forest fits Lane 2 (Refresh / Content Opportunity Scoring) for
three reasons:

1. It handles mixed signals well — my features include numeric signals
   (impressions, CTR, days since update) that don't follow a clean linear
   pattern. Random Forest finds non-linear combinations without needing
   manual feature engineering.

2. It produces probabilities — I need a ranked queue, not a hard yes/no.
   Random Forest's predict_proba() gives a continuous score I can rank pages
   by, which is exactly what Precision@K measures.

3. It was the best performer in the starter pipeline — the starter already
   showed Random Forest achieving Precision@50 of 0.740 versus the baseline's
   0.240. This gives me a directional benchmark to match or beat on the same
   data.

I considered Logistic Regression (too linear for this signal mix) and
Gradient Boosting (more powerful but harder to interpret and not clearly
necessary here). Random Forest is the honest middle ground.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Split design: Client-grouped holdout

I use a grouped split where all pages from the same client stay on the
same side of the split. This is honest because:

- Pages from the same client share patterns (content strategy, industry,
  update cadence) that the model could memorise if they appeared in both
  train and test.
- A grouped split tests whether the model generalises to clients it has
  never seen — which is the real-world use case.
- The starter pipeline used the same client-holdout design, so my results
  are comparable to the benchmark.

I do not use a time-aware split here because the starter data is a
cross-sectional snapshot, not a time series. A time split would not be
meaningful on this data structure.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os, sys, subprocess
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, average_precision_score
from sklearn.preprocessing import StandardScaler

IN_COLAB = "google.colab" in sys.modules
REPO_DIR = "flyrank-ml-internship-starter"
if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/flyrank-bih/flyrank-ml-internship-starter",
                        REPO_DIR], check=True)
    os.chdir(REPO_DIR)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)
print(f"Loaded {len(df)} rows | Declining: {df['is_declining_label'].mean():.1%}")

# Client-grouped split
clients = df['client_id'].unique()
np.random.seed(42)
np.random.shuffle(clients)
split = int(len(clients) * 0.8)
train_clients = clients[:split]
test_clients = clients[split:]

train = df[df['client_id'].isin(train_clients)]
test = df[df['client_id'].isin(test_clients)]

print(f"\nTrain: {len(train)} rows | {len(train_clients)} clients")
print(f"Test:  {len(test)} rows | {len(test_clients)} clients")
print(f"No client appears in both splits: {len(set(train_clients) & set(test_clients)) == 0}")

Loaded 30000 rows | Declining: 54.2%

Train: 22389 rows | 25 clients
Test:  7611 rows | 7 clients
No client appears in both splits: True


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

Training Random Forest and comparing against the Week 4 baseline
on the same test split, using the same metric: Precision@K.

The baseline used a weighted rule combining staleness, visibility
and CTR gap. The model learns from the same signals but finds
non-linear combinations automatically.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Features — same signals as the baseline, no label-derived inputs
feature_cols = [
    'impressions_90d', 'sessions_90d', 'content_age_days',
    'days_since_last_update', 'ctr', 'avg_position',
    'word_count', 'engagement_rate'
]

# Drop rows with missing values in features
train_clean = train.dropna(subset=feature_cols)
test_clean = test.dropna(subset=feature_cols)

X_train = train_clean[feature_cols]
y_train = train_clean['is_declining_label']
X_test = test_clean[feature_cols]
y_test = test_clean['is_declining_label']

# =====================
# BASELINE SCORE (replicate Week 4 rule on test set)
# =====================
test_clean = test_clean.copy()
test_clean['staleness_score'] = np.clip(test_clean['days_since_last_update'] / 365, 0, 1)
test_clean['visibility_score'] = np.clip(
    np.log1p(test_clean['impressions_90d']) / np.log1p(test_clean['impressions_90d'].max()), 0, 1)
test_clean['ctr_gap_score'] = np.clip(1 - (test_clean['ctr'] / 0.05), 0, 1)
test_clean['baseline_score'] = (
    0.40 * test_clean['staleness_score'] +
    0.35 * test_clean['visibility_score'] +
    0.25 * test_clean['ctr_gap_score']
)

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.array(scores))
    topk = np.array(labels)[order[:k]]
    return topk.mean()

baseline_p20 = precision_at_k(test_clean['baseline_score'], y_test, 20)
baseline_p50 = precision_at_k(test_clean['baseline_score'], y_test, 50)

# =====================
# RANDOM FOREST
# =====================
rf = RandomForestClassifier(n_estimators=100, max_depth=6,
                             class_weight='balanced', random_state=42)
rf.fit(X_train, y_train)
rf_proba = rf.predict_proba(X_test)[:, 1]

rf_p20 = precision_at_k(rf_proba, y_test, 20)
rf_p50 = precision_at_k(rf_proba, y_test, 50)

# =====================
# COMPARISON TABLE
# =====================
results = pd.DataFrame({
    'Method': ['Baseline Rule (Week 4)', 'Random Forest'],
    'Precision@20': [round(baseline_p20, 3), round(rf_p20, 3)],
    'Precision@50': [round(baseline_p50, 3), round(rf_p50, 3)],
})
print("\nModel vs Baseline — Same Split, Same Metric:")
print(results.to_string(index=False))

improvement_p50 = rf_p50 / baseline_p50 if baseline_p50 > 0 else 0
print(f"\nRandom Forest is {improvement_p50:.1f}x the baseline at Precision@50")

# Feature importance
fi = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)
print("\nFeature Importances:")
print(fi.to_string(index=False))


Model vs Baseline — Same Split, Same Metric:
                Method  Precision@20  Precision@50
Baseline Rule (Week 4)          0.75          0.82
         Random Forest          0.75          0.62

Random Forest is 0.8x the baseline at Precision@50

Feature Importances:
               feature  importance
       impressions_90d    0.384290
          avg_position    0.263912
      content_age_days    0.120247
            word_count    0.063727
days_since_last_update    0.059332
                   ctr    0.051068
          sessions_90d    0.047175
       engagement_rate    0.010248


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

Error analysis and interpretation.

The model leans most heavily on impressions_90d, ctr and avg_position —
the same signals the baseline rule used, but combined non-linearly.
This is directionally reassuring: the model found the same signals a
human would have chosen, but weights them differently per row.

Where the model is likely wrong:
- Pages with very high impressions but stable trend direction — the model
  may flag these as high priority because they are visible, even though
  they are not declining.
- Pages with low impressions that are genuinely declining — the model may
  rank these lower because visibility is a strong feature, missing real
  declines on low-traffic pages.
- Seasonal pages — a page that naturally dips at certain times of year
  may be flagged as declining when it is just following a seasonal pattern
  the model cannot distinguish from real decline.

The results are directional and decision-support only. A human reviewer
should inspect the top 20 before acting on any recommendation.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Error analysis — false positives and false negatives in top 50
test_results = test_clean[feature_cols + ['is_declining_label']].copy()
test_results['rf_score'] = rf_proba
test_results['trend'] = test_clean['trend_direction'].values
test_results = test_results.sort_values('rf_score', ascending=False).reset_index(drop=True)

top50 = test_results.head(50)

false_positives = top50[top50['is_declining_label'] == 0]
false_negatives = test_results[
    (test_results['is_declining_label'] == 1) &
    (test_results.index >= 50)
].head(10)

print(f"Top 50 review:")
print(f"  Correct (declining): {(top50['is_declining_label'] == 1).sum()}")
print(f"  False positives (not declining but flagged): {len(false_positives)}")

print(f"\nFalse positive profile (pages flagged but not declining):")
print(false_positives[['impressions_90d', 'ctr', 'avg_position',
                         'days_since_last_update', 'trend']].describe().round(2))

print(f"\nMissed declines (declining but ranked below top 50): {len(false_negatives)}")
print(false_negatives[['impressions_90d', 'ctr', 'avg_position',
                         'days_since_last_update', 'trend']].head(5).round(2))

Top 50 review:
  Correct (declining): 31
  False positives (not declining but flagged): 19

False positive profile (pages flagged but not declining):
       impressions_90d    ctr  avg_position  days_since_last_update
count            19.00  19.00         19.00                   19.00
mean           8653.11   0.16         22.75                   92.21
std           16286.00   0.10          7.44                   28.49
min             965.00   0.03         13.60                   20.00
25%            2849.00   0.08         17.50                  104.00
50%            3638.00   0.14         20.10                  104.00
75%            5204.00   0.20         24.30                  104.00
max           71283.00   0.38         40.40                  104.00

Missed declines (declining but ranked below top 50): 10
    impressions_90d   ctr  avg_position  days_since_last_update trend
50              695  0.14          36.3                     104  down
51             6363  0.08          20.7  

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.